### Libraries

In [ ]:
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")


In [ ]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
# Import the required Libraries
import numpy as np
import math
# from osgeo import gdal
import matplotlib.pyplot as plt
import os
import matplotlib.patches as mpatches
# from skimage import exposure
# from skimage.io import imread, imshow,imsave
# from tqdm import tqdm_notebook as tqdm
# import tensorflow as tf
from tensorflow.python.keras import backend as K
from keras.models import *
from keras.layers import *
from keras.optimizers import *
# import pandas as pd
# from prettytable import PrettyTable
from keras import callbacks
import time
# from pycocotools import mask
# from skimage import measure
# import cv2
# from sklearn.model_selection import train_test_split
from decimal import *
from matplotlib.colors import ListedColormap,to_rgb
from matplotlib.patches import Patch
from keras.metrics import MeanIoU
# import sklearn
# import simple_colors
# import scipy.io
from tensorflow.keras.utils import to_categorical
from keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D, Reshape, Dense, multiply, Permute, Concatenate, Conv2D, Add, Activation, Lambda
from keras import backend as K
from keras.activations import sigmoid
from keras import models, layers, regularizers
from keras.regularizers import l2
# import segmentation_models as sm
from datetime import datetime

In [ ]:
#Mohammad Library:
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import random
import numpy as np
from tensorflow.keras.layers import concatenate, Input
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import random
import numpy as np
from tensorflow.keras import layers
from tensorflow.keras.layers import concatenate, Input
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Add, concatenate, Activation
from tensorflow.keras.models import Model
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import GRU
from tensorflow.keras.layers import RNN
from tensorflow import keras
from tensorflow.keras.layers import Dense, Add
from tensorflow.keras.layers import Flatten
from tensorflow.keras import backend as K
from tensorflow.keras.layers import TimeDistributed
import os
import shutil
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Add, concatenate, Activation
from tensorflow.keras.models import Model
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import GRU
from tensorflow.keras.layers import RNN
from tensorflow import keras
from tensorflow.keras.layers import Dense, Add
from tensorflow.keras.layers import Flatten
from tensorflow.keras import backend as K
from tensorflow.keras.layers import TimeDistributed
import os
import shutil
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

### Model Architecture

In [ ]:
import math
import tensorflow as tf
from keras import layers
from keras.utils.generic_utils import register_keras_serializable
from keras.utils.tf_utils import shape_type_conversion
# from tfswin.norm import LayerNorm
from tensorflow.nn import depth_to_space


# @register_keras_serializable(package='TFSwinV2')
class PatchExpanding(layers.Layer):
    def __init__(self, swin_v2=False, name = '', upsample_rate=2,return_vector=True, **kwargs):
        super().__init__(**kwargs)
        self.input_spec = layers.InputSpec(ndim=4)
        self.return_vector = return_vector
        self.swin_v2 = swin_v2
        self.upsample_rate = upsample_rate


    @shape_type_conversion
    def build(self, input_shape):
        # noinspection PyAttributeOutsideInit
        self.channels = input_shape[-1]
        self.H = input_shape[1]
        self.W = input_shape[2]
        if self.channels is None:
            raise ValueError('Channel dimensions of the inputs should be defined. Found `None`.')

        self.input_spec = layers.InputSpec(ndim=4, axes={-1: self.channels})

        # noinspection PyAttributeOutsideInit
        self.norm = LayerNorm(name='norm')
        # Linear transformations that doubles the channels
        self.linear_trans1 = layers.Conv2D(self.upsample_rate*self.channels, kernel_size=1, use_bias=False, name='{}_linear_trans1'.format(self.name))
        #
        #self.linear_trans2 = layers.Conv2D(self.upsample_rate*self.channels, kernel_size=1, use_bias=False, name='{}_linear_trans1'.format(name))
        self.prefix = self.name


        super().build(input_shape)

    def call(self, inputs, *args, **kwargs):
        _, H, W, C = inputs.get_shape().as_list()
        x = self.linear_trans1(inputs)

        x = depth_to_space(x, self.upsample_rate, data_format='NHWC', name='{}_d_to_space'.format(self.prefix))

        if self.return_vector:
            # Convert aligned patches to a patch sequence
            x = tf.reshape(x, (-1, H*W*self.upsample_rate*self.upsample_rate, C//2))

        return x

    @shape_type_conversion
    def compute_output_shape(self, input_shape):
        def _scale(value):
            return None if value is None else math.ceil(value * 2)

        return input_shape[0], _scale(input_shape[1]), _scale(input_shape[2]), self.channels / 2

    def get_config(self):
        config = super().get_config()
        config.update({'swin_v2': self.swin_v2,
        'name': self.name,
        'return_vector':self.return_vector,
        'upsample_rate':self.upsample_rate
        })

        return config


In [ ]:
import numpy as np
import tensorflow as tf
from keras import initializers, layers
from keras.utils.control_flow_util import smart_cond
from keras.utils.generic_utils import register_keras_serializable
from keras.utils.tf_utils import shape_type_conversion


# @register_keras_serializable(package='TFSwinV2')
class WindowAttention(layers.Layer):
    def __init__(self, num_heads, qkv_bias=True, qk_scale=None, attn_drop=0., proj_drop=0.,
                 window_pretrain=0, swin_v2=False, **kwargs):
        super().__init__(**kwargs)
        self.input_spec = [
            layers.InputSpec(ndim=3), layers.InputSpec(ndim=0, dtype='int32'), layers.InputSpec(ndim=1, dtype='int32'),
            layers.InputSpec(ndim=5), layers.InputSpec(ndim=0, dtype='bool')]

        self.num_heads = num_heads
        self.qkv_bias = qkv_bias
        self.qk_scale = qk_scale
        self.attn_drop = attn_drop
        self.proj_drop = proj_drop
        self.window_pretrain = window_pretrain
        self.swin_v2 = swin_v2

    @shape_type_conversion
    def build(self, input_shape):
        # noinspection PyAttributeOutsideInit
        self.channels = input_shape[0][-1]
        if self.channels is None:
            raise ValueError('Channel dimensions of the inputs should be defined. Found `None`.')

        qkv_bias = not self.swin_v2 and self.qkv_bias
        # noinspection PyAttributeOutsideInit
        self.qkv = layers.Dense(self.channels * 3, use_bias=qkv_bias, name='qkv')

        if self.swin_v2:
            # noinspection PyAttributeOutsideInit
            self.scale = self.add_weight(
                'logit_scale',
                shape=[self.num_heads, 1, 1],
                initializer=initializers.Constant(np.log(10.)),
                trainable=True,
                dtype=self.dtype)

            # noinspection PyAttributeOutsideInit
            self.cpb0 = layers.Dense(512, activation='relu', name='cpb_mlp.0')
            self.cpb1 = layers.Dense(self.num_heads, activation='sigmoid', use_bias=False, name=f'cpb_mlp.2')

            # noinspection PyAttributeOutsideInit
            self.q_bias = None
            # noinspection PyAttributeOutsideInit
            self.v_bias = None
            if self.qkv_bias:
                self.q_bias = self.add_weight(
                    'q_bias',
                    shape=[self.channels],
                    initializer='zeros',
                    trainable=True,
                    dtype=self.dtype)
                self.v_bias = self.add_weight(
                    'v_bias',
                    shape=[self.channels],
                    initializer='zeros',
                    trainable=True,
                    dtype=self.dtype)
        else:
            # noinspection PyAttributeOutsideInit
            self.scale = self.qk_scale or (self.channels // self.num_heads) ** -0.5

            # noinspection PyAttributeOutsideInit
            self.relative_bias = self.add_weight(
                'relative_position_bias_table',
                shape=[(2 * self.window_pretrain - 1) ** 2, self.num_heads],
                initializer=initializers.TruncatedNormal(stddev=0.02),
                trainable=True,
                dtype=self.dtype)

        # noinspection PyAttributeOutsideInit
        self.drop_attn = layers.Dropout(self.attn_drop)

        # noinspection PyAttributeOutsideInit
        self.proj = layers.Dense(self.channels, name='proj')

        # noinspection PyAttributeOutsideInit
        self.drop_proj = layers.Dropout(self.proj_drop)

        super().build(input_shape)

    def relative_table(self, window_size):
        offset = tf.range(1 - window_size, window_size)
        offset = tf.cast(offset, self.compute_dtype)
        offset = tf.stack(tf.meshgrid(offset, offset, indexing='ij'))
        offset = tf.transpose(offset, [1, 2, 0])[None]

        window = self.window_pretrain if self.window_pretrain > 0 else window_size

        offset *= 8. / (tf.cast(window, self.compute_dtype) - 1.)
        offset = tf.sign(offset) * tf.math.log1p(tf.abs(offset)) / np.log(8)

        return offset

    def with_mask(self, attn, mask, length):
        mask_windows = tf.shape(mask)[1]
        attn = tf.reshape(attn, shape=[-1, mask_windows, self.num_heads, length, length])
        attn += mask
        attn = tf.reshape(attn, shape=[-1, self.num_heads, length, length])

        return attn

    def call(self, inputs, **kwargs):
        inputs, window_size, relative_index, attention_mask, with_mask = inputs
        length = tf.shape(inputs)[1]

        qkv = self.qkv(inputs)
        if self.swin_v2 and self.qkv_bias:
            k_bias = tf.zeros_like(self.v_bias, self.compute_dtype)
            qkv_bias = tf.concat([self.q_bias, k_bias, self.v_bias], axis=0)
            qkv = tf.nn.bias_add(qkv, qkv_bias)
        qkv = tf.reshape(qkv, [-1, length, 3, self.num_heads, self.channels // self.num_heads])
        qkv = tf.transpose(qkv, [2, 0, 3, 1, 4])

        q, k, v = tf.unstack(qkv, 3)
        if self.swin_v2:
            scale = tf.minimum(self.scale, np.log(1. / .01))
            scale = tf.exp(scale)
            q, _ = tf.linalg.normalize(q, axis=-1)
            k, _ = tf.linalg.normalize(k, axis=-1)
        else:
            scale = self.scale
        q *= scale
        attn = tf.matmul(q, k, transpose_b=True)

        if self.swin_v2:
            relative_bias = self.cpb0(self.relative_table(window_size))
            relative_bias = self.cpb1(relative_bias)
            relative_bias = tf.reshape(relative_bias, [-1, self.num_heads])
            bias = tf.gather(relative_bias, relative_index) * 16.
        else:
            bias = tf.gather(self.relative_bias, relative_index)
        bias = tf.reshape(bias, [window_size ** 2, window_size ** 2, -1])
        bias = tf.transpose(bias, perm=[2, 0, 1])
        attn = attn + bias[None]

        attn = smart_cond(
            with_mask,
            lambda: self.with_mask(attn, attention_mask, length),
            lambda: tf.identity(attn))

        attn = tf.nn.softmax(attn)
        attn = self.drop_attn(attn)

        outputs = tf.transpose(tf.matmul(attn, v), perm=[0, 2, 1, 3])
        outputs = tf.reshape(outputs, [-1, length, self.channels])

        outputs = self.proj(outputs)
        outputs = self.drop_proj(outputs)

        return outputs

    @shape_type_conversion
    def compute_output_shape(self, input_shape):
        return input_shape[0]

    def get_config(self):
        config = super().get_config()

        config.update({
            'num_heads': self.num_heads,
            'qkv_bias': self.qkv_bias,
            'qk_scale': self.qk_scale,
            'attn_drop': self.attn_drop,
            'proj_drop': self.proj_drop,
            'window_pretrain': self.window_pretrain,
            'swin_v2': self.swin_v2
        })

        return config


In [ ]:
import tensorflow as tf
from keras import layers
from keras.utils.generic_utils import register_keras_serializable
from keras.utils.tf_utils import shape_type_conversion
# from tfswin.norm import LayerNorm


@register_keras_serializable(package='TFSwinV2')
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size, embed_dim, normalize, **kwargs):
        super().__init__(**kwargs)
        self.input_spec = layers.InputSpec(ndim=4)

        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.normalize = normalize

    @shape_type_conversion
    def build(self, input_shape):
        # noinspection PyAttributeOutsideInit
        self.proj = layers.Conv2D(
            self.embed_dim, kernel_size=self.patch_size, strides=self.patch_size, padding='same', name='proj')

        if self.normalize:
            # noinspection PyAttributeOutsideInit
            self.norm = LayerNorm(name='norm')

        super().build(input_shape)

    def call(self, inputs, *args, **kwargs):
        outputs = self.proj(inputs)

        if self.normalize:
            outputs = self.norm(outputs)

        return outputs

    @shape_type_conversion
    def compute_output_shape(self, input_shape):
        return self.proj.compute_output_shape(input_shape)

    def get_config(self):
        config = super().get_config()
        config.update({
            'patch_size': self.patch_size,
            'embed_dim': self.embed_dim,
            'normalize': self.normalize,
        })

        return config


In [ ]:
import math
import tensorflow as tf
from keras import layers
from keras.utils.generic_utils import register_keras_serializable
from keras.utils.tf_utils import shape_type_conversion
# from tfswin.norm import LayerNorm


@register_keras_serializable(package='TFSwinV2')
class PatchMerging(layers.Layer):
    def __init__(self, swin_v2=False, **kwargs):
        super().__init__(**kwargs)
        self.input_spec = layers.InputSpec(ndim=4)

        self.swin_v2 = swin_v2

    @shape_type_conversion
    def build(self, input_shape):
        # noinspection PyAttributeOutsideInit
        self.channels = input_shape[-1]
        if self.channels is None:
            raise ValueError('Channel dimensions of the inputs should be defined. Found `None`.')
        self.input_spec = layers.InputSpec(ndim=4, axes={-1: self.channels})

        # noinspection PyAttributeOutsideInit
        self.norm = LayerNorm(name='norm')

        # noinspection PyAttributeOutsideInit
        self.reduction = layers.Dense(self.channels * 2, use_bias=False, name='reduction')

        super().build(input_shape)

    def call(self, inputs, *args, **kwargs):
        paddings = [[0, 0], [0, 1], [0, 1], [0, 0]]
        outputs = tf.pad(inputs, paddings)

        slice00 = outputs[:, 0:-1:2, 0:-1:2, :]  # B H/2 W/2 C
        slice10 = outputs[:, 1::2, 0:-1:2, :]  # B H/2 W/2 C
        slice01 = outputs[:, 0:-1:2, 1::2, :]  # B H/2 W/2 C
        slice11 = outputs[:, 1::2, 1::2, :]  # B H/2 W/2 C
        outputs = tf.concat([slice00, slice10, slice01, slice11], axis=-1)

        if self.swin_v2:
            outputs = self.reduction(outputs)
            outputs = self.norm(outputs)
        else:
            outputs = self.norm(outputs)
            outputs = self.reduction(outputs)

        return outputs

    @shape_type_conversion
    def compute_output_shape(self, input_shape):
        def _scale(value):
            return None if value is None else math.ceil(value / 2)

        return input_shape[0], _scale(input_shape[1]), _scale(input_shape[2]), self.channels * 2

    def get_config(self):
        config = super().get_config()
        config.update({'swin_v2': self.swin_v2})

        return config


In [ ]:
import tensorflow as tf
import warnings
from keras import layers
from keras.utils.generic_utils import register_keras_serializable


@register_keras_serializable(package='TFSwinV2')
class LayerNorm(layers.LayerNormalization):
    # Overload defaults and casting to use fused implementation

    def __init__(self, epsilon=1.001e-5, dtype='float32', **kwargs):
        kwargs['autocast'] = False
        super().__init__(epsilon=epsilon, dtype=dtype, **kwargs)

    def build(self, input_shape):
        super().build(input_shape)
        if not self._fused:
            warnings.warn(f'Layer {self.name} will use an inefficient implementation.')

    def call(self, inputs, *args, **kwargs):
        outputs = tf.cast(inputs, 'float32')

        outputs = super().call(outputs)

        if inputs.dtype == tf.dtypes.float16:
            outputs = tf.clip_by_value(outputs, tf.dtypes.float16.min, tf.dtypes.float16.max)
        outputs = tf.cast(outputs, inputs.dtype)

        return outputs

    def compute_output_signature(self, input_signature):
        return input_signature


In [ ]:
import tensorflow as tf
from keras.applications import imagenet_utils


def preprocess_input(inputs):
    inputs = tf.cast(inputs, 'float32')
    outputs = imagenet_utils.preprocess_input(inputs, data_format='channels_last', mode='torch')

    return outputs


In [ ]:
import tensorflow as tf
from keras import backend, layers
from keras.utils.control_flow_util import smart_cond
from keras.utils.generic_utils import register_keras_serializable
from keras.utils.tf_utils import shape_type_conversion


@register_keras_serializable(package='TFSwinV2')
class DropPath(layers.Layer):
    def __init__(self, rate, **kwargs):
        super().__init__(**kwargs)
        self.input_spec = layers.InputSpec(min_ndim=1)

        if not 0. <= rate <= 1.:
            raise ValueError(f'Invalid value {rate} received for `rate`. Expected a value between 0 and 1.')

        self.rate = rate

    def call(self, inputs, training=None, **kwargs):
        if 0. == self.rate:
            return inputs

        if training is None:
            training = backend.learning_phase()

        outputs = smart_cond(training, lambda: self.drop(inputs), lambda: tf.identity(inputs))

        return outputs

    def drop(self, inputs):
        keep = 1.0 - self.rate
        batch = tf.shape(inputs)[0]
        shape = [batch] + [1] * (inputs.shape.rank - 1)

        random = tf.random.uniform(shape, dtype=self.compute_dtype) <= keep
        random = tf.cast(random, self.compute_dtype) / keep

        outputs = inputs * random

        return outputs

    @shape_type_conversion
    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        config = super().get_config()
        config.update({'rate': self.rate})

        return config

In [ ]:
import tensorflow as tf


def window_partition(inputs, height, width, window_size, dtype=None, name=None):
    with tf.name_scope(name or 'window_partition'):
        inputs = tf.convert_to_tensor(inputs, dtype)

        if 4 != inputs.shape.rank:
            raise ValueError('Expecting inputs rank to be 4.')

        channels = inputs.shape[-1]
        if channels is None:
            raise ValueError('Channel dimensions of the inputs should be defined. Found `None`.')

        windows_height = height // window_size
        windows_width = width // window_size

        outputs = tf.reshape(inputs, [-1, windows_height, window_size, windows_width, window_size, channels])
        outputs = tf.transpose(outputs, [0, 1, 3, 2, 4, 5])
        outputs = tf.reshape(outputs, [-1, window_size ** 2, channels])

        return outputs


def window_reverse(inputs, height, width, window_size, dtype=None, name=None):
    with tf.name_scope(name or 'window_reverse'):
        inputs = tf.convert_to_tensor(inputs, dtype)

        if 3 != inputs.shape.rank:
            raise ValueError('Expecting inputs rank to be 3.')

        channels = inputs.shape[-1]
        if channels is None:
            raise ValueError('Channel dimensions of the inputs should be defined. Found `None`.')

        windows_height = height // window_size
        windows_width = width // window_size

        outputs = tf.reshape(inputs, [-1, windows_height, windows_width, window_size, window_size, channels])
        outputs = tf.transpose(outputs, [0, 1, 3, 2, 4, 5])
        outputs = tf.reshape(outputs, [-1, height, width, channels])

        return outputs


In [ ]:
import tensorflow as tf
from keras import initializers, layers
from keras.utils.generic_utils import register_keras_serializable
from keras.utils.tf_utils import shape_type_conversion


@register_keras_serializable(package='TFSwinV2')
class AbsoluteEmbedding(layers.Layer):
    def __init__(self, pretrain_size, **kwargs):
        super().__init__(**kwargs)
        self.input_spec = layers.InputSpec(ndim=4)

        self.pretrain_size = pretrain_size

    @shape_type_conversion
    def build(self, input_shape):
        channels = input_shape[-1]
        if channels is None:
            raise ValueError('Channel dimension of the inputs should be defined. Found `None`.')
        self.input_spec = layers.InputSpec(ndim=4, axes={-1: channels})

        # noinspection PyAttributeOutsideInit
        self.embedding = self.add_weight(
            'embedding',
            shape=[1, self.pretrain_size, self.pretrain_size, channels],
            initializer=initializers.TruncatedNormal(stddev=0.02),
            trainable=True,
            dtype=self.dtype)

        super().build(input_shape)

    def call(self, inputs, *args, **kwargs):
        new_size = tf.shape(inputs)[1:3]
        embeddings = tf.image.resize(self.embedding, new_size, method=tf.image.ResizeMethod.BICUBIC)
        embeddings = tf.cast(embeddings, inputs.dtype)

        return inputs + embeddings

    @shape_type_conversion
    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        config = super().get_config()
        config.update({'pretrain_size': self.pretrain_size})

        return config


In [ ]:
import tensorflow as tf
from keras import layers
from keras.utils.control_flow_util import smart_cond
from keras.utils.generic_utils import register_keras_serializable
from keras.utils.tf_utils import shape_type_conversion
# from tfswin.drop import DropPath
# from tfswin.mlp import MLP
# from tfswin.norm import LayerNorm
# from tfswin.winatt import WindowAttention
# from tfswin.window import window_partition, window_reverse


@register_keras_serializable(package='TFSwinV2')
class SwinBlock(layers.Layer):
    def __init__(self, num_heads, mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0., path_drop=0.,
                 window_pretrain=0, swin_v2=False, **kwargs):
        super().__init__(**kwargs)
        self.input_spec = [
            layers.InputSpec(ndim=4), layers.InputSpec(ndim=0, dtype='int32'), layers.InputSpec(ndim=0, dtype='int32'),
            layers.InputSpec(ndim=1, dtype='int32'), layers.InputSpec(ndim=5)]
        self.num_heads = num_heads
        self.mlp_ratio = mlp_ratio
        self.qkv_bias = qkv_bias
        self.qk_scale = qk_scale
        self.drop = drop
        self.attn_drop = attn_drop
        self.path_drop = path_drop
        self.window_pretrain = window_pretrain
        self.swin_v2 = swin_v2

    @shape_type_conversion
    def build(self, input_shape):
        norm_init = 'zeros' if self.swin_v2 else 'ones'

        # noinspection PyAttributeOutsideInit
        self.norm1 = LayerNorm(gamma_initializer=norm_init, name='norm1')

        # noinspection PyAttributeOutsideInit
        self.attn = WindowAttention(num_heads=self.num_heads, qkv_bias=self.qkv_bias, qk_scale=self.qk_scale,
                                    attn_drop=self.attn_drop, proj_drop=self.drop, window_pretrain=self.window_pretrain,
                                    swin_v2=self.swin_v2, name='attn')

        # noinspection PyAttributeOutsideInit
        self.drop_path = DropPath(self.path_drop)

        # noinspection PyAttributeOutsideInit
        self.norm2 = LayerNorm(gamma_initializer=norm_init, name='norm2')

        # noinspection PyAttributeOutsideInit
        self.mlp = MLP(ratio=self.mlp_ratio, dropout=self.drop, name='mlp')

        super().build(input_shape)

    def call(self, inputs, *args, **kwargs):
        inputs, shift_size, window_size, relative_index, attention_mask = inputs
        height, width = tf.unstack(tf.shape(inputs)[1:3])

        with_shift = tf.greater(shift_size, 0)

        if self.swin_v2:
            outputs = inputs
        else:
            outputs = self.norm1(inputs)

        h_pad = (window_size - height % window_size) % window_size
        w_pad = (window_size - width % window_size) % window_size
        paddings = [[0, 0], [0, h_pad], [0, w_pad], [0, 0]]
        outputs = tf.pad(outputs, paddings)
        padded_height, padded_width = height + h_pad, width + w_pad

        # Cyclic shift
        outputs = smart_cond(
            with_shift,
            lambda: tf.roll(outputs, [-shift_size, -shift_size], [1, 2]),
            lambda: tf.identity(outputs))
        if tf.executing_eagerly():
            pass

        # Partition windows
        outputs = window_partition(outputs, padded_height, padded_width, window_size, self.compute_dtype)

        # W-MSA/SW-MSA
        outputs = self.attn([outputs, window_size, relative_index, attention_mask, with_shift])

        # Merge windows
        outputs = window_reverse(outputs, padded_height, padded_width, window_size, self.compute_dtype)

        # Reverse cyclic shift
        outputs = smart_cond(
            with_shift,
            lambda: tf.roll(outputs, [shift_size, shift_size], [1, 2]),
            lambda: tf.identity(outputs)
        )

        outputs = outputs[:, :height, :width, ...]

        # FFN
        if self.swin_v2:
            outputs = inputs + self.drop_path(self.norm1(outputs))
            outputs += self.drop_path(self.norm2(self.mlp(outputs)))
        else:
            outputs = inputs + self.drop_path(outputs)
            outputs += self.drop_path(self.mlp(self.norm2(outputs)))

        return outputs

    @shape_type_conversion
    def compute_output_shape(self, input_shape):
        return input_shape[0]

    def get_config(self):
        config = super().get_config()
        config.update({
            'num_heads': self.num_heads,
            'mlp_ratio': self.mlp_ratio,
            'qkv_bias': self.qkv_bias,
            'qk_scale': self.qk_scale,
            'drop': self.drop,
            'attn_drop': self.attn_drop,
            'path_drop': self.path_drop,
            'window_pretrain': self.window_pretrain,
            'swin_v2': self.swin_v2
        })

        return config


In [ ]:
from keras import activations, layers
from keras.utils.generic_utils import register_keras_serializable
from keras.utils.tf_utils import shape_type_conversion


@register_keras_serializable(package='TFSwinV2')
class MLP(layers.Layer):
    def __init__(self, ratio, dropout, **kwargs):
        super().__init__(**kwargs)
        self.input_spec = layers.InputSpec(ndim=4)

        self.ratio = ratio
        self.dropout = dropout

    @shape_type_conversion
    def build(self, input_shape):
        channels = input_shape[-1]
        if channels is None:
            raise ValueError('Channel dimension of the inputs should be defined. Found `None`.')
        self.input_spec = layers.InputSpec(ndim=4, axes={-1: channels})

        # noinspection PyAttributeOutsideInit
        self.fc1 = layers.Dense(int(channels * self.ratio), name='fc1')

        # noinspection PyAttributeOutsideInit
        self.fc2 = layers.Dense(channels, name='fc2')

        # noinspection PyAttributeOutsideInit
        self.drop = layers.Dropout(self.dropout)

        super().build(input_shape)

    def call(self, inputs, *args, **kwargs):
        outputs = self.fc1(inputs)
        outputs = activations.gelu(outputs)
        outputs = self.drop(outputs)
        outputs = self.fc2(outputs)
        outputs = self.drop(outputs)

        return outputs

    @shape_type_conversion
    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        config = super().get_config()
        config.update({
            'ratio': self.ratio,
            'dropout': self.dropout
        })

        return config


In [ ]:
import numpy as np
import tensorflow as tf
from keras import layers
from keras.utils.control_flow_util import smart_cond
from keras.utils.generic_utils import register_keras_serializable
from keras.utils.tf_utils import shape_type_conversion
# from tfswin.swin import SwinBlock
# from tfswin.window import window_partition


@register_keras_serializable(package='TFSwinV2')
class BasicLayer(layers.Layer):
    def __init__(self, depth, num_heads, window_size, mlp_ratio=4., qkv_bias=True, qk_scale=None,
                 drop=0., attn_drop=0., path_drop=0., window_pretrain=0, swin_v2=False, **kwargs):
        super().__init__(**kwargs)
        self.input_spec = layers.InputSpec(ndim=4)

        self.depth = depth
        self.num_heads = num_heads
        self.window_size = window_size
        self.mlp_ratio = mlp_ratio
        self.qkv_bias = qkv_bias
        self.qk_scale = qk_scale
        self.drop = drop
        self.attn_drop = attn_drop
        self.path_drop = path_drop
        self.window_pretrain = window_pretrain
        self.swin_v2 = swin_v2

        self.shift_size = self.window_size // 2

    @shape_type_conversion
    def build(self, input_shape):
        path_drop = self.path_drop
        if not isinstance(self.path_drop, (list, tuple)):
            path_drop = [self.path_drop] * self.depth

        shift_size = np.zeros(self.depth, 'int32')
        shift_size[1::2] = self.shift_size

        # noinspection PyAttributeOutsideInit
        self.blocks = [
            SwinBlock(num_heads=self.num_heads, mlp_ratio=self.mlp_ratio, qkv_bias=self.qkv_bias,
                      qk_scale=self.qk_scale, drop=self.drop, attn_drop=self.attn_drop, path_drop=path_drop[i],
                      window_pretrain=self.window_pretrain, swin_v2=self.swin_v2, name=f'blocks.{i}')
            for i in range(self.depth)]

        super().build(input_shape)

    def shift_window(self, height, width):
        min_size = tf.minimum(height, width)
        shift_size, window_size = smart_cond(
            tf.less_equal(min_size, self.window_size),
            lambda: (0, min_size),
            lambda: (self.shift_size, self.window_size))

        return shift_size, window_size

    def relative_index(self, window_size):
        offset = tf.range(window_size)
        offset = tf.stack(tf.meshgrid(offset, offset, indexing='ij'), axis=0)
        offset = tf.reshape(offset, [2, -1])
        offset = offset[:, :, None] - offset[:, None]

        index = offset + (window_size - 1)
        index = index[0] * (2 * window_size - 1) + index[1]
        index = tf.reshape(index, [-1])

        return index

    def attention_mask(self, height, width, window_size):
        padded_height = tf.cast(tf.math.ceil(height / window_size), 'int32') * window_size
        padded_width = tf.cast(tf.math.ceil(width / window_size), 'int32') * window_size

        last_repeats = [window_size - self.shift_size, self.shift_size]

        image_mask = np.arange(9, dtype='int32').reshape((3, 3))
        image_mask = tf.repeat(image_mask, [padded_height - window_size] + last_repeats, axis=1)
        image_mask = tf.repeat(image_mask, [padded_width - window_size] + last_repeats, axis=0)
        image_mask = image_mask[None, ..., None]

        mask_windows = window_partition(image_mask, padded_height, padded_width, window_size, 'int32')
        mask_windows = mask_windows[..., 0]

        attn_mask = mask_windows[:, None] - mask_windows[:, :, None]
        attn_mask = tf.where(attn_mask == 0, 0., -100.)
        attn_mask = tf.cast(attn_mask, self.compute_dtype)
        attn_mask = attn_mask[None, :, None, ...]

        return attn_mask

    def call(self, inputs, *args, **kwargs):
        height, width = tf.unstack(tf.shape(inputs)[1:3])

        shift_size, window_size = self.shift_window(height, width)
        relative_index = self.relative_index(window_size)
        attention_mask = self.attention_mask(height, width, window_size)

        outputs = inputs
        for i, b in enumerate(self.blocks):
            current_shift = shift_size if i % 2 else 0
            outputs = b([outputs, current_shift, window_size, relative_index, attention_mask])

        return outputs

    @shape_type_conversion
    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        config = super().get_config()
        config.update({
            'depth': self.depth,
            'num_heads': self.num_heads,
            'window_size': self.window_size,
            'mlp_ratio': self.mlp_ratio,
            'qkv_bias': self.qkv_bias,
            'qk_scale': self.qk_scale,
            'drop': self.drop,
            'attn_drop': self.attn_drop,
            'path_drop': self.path_drop,
            'window_pretrain': self.window_pretrain,
            'swin_v2': self.swin_v2
        })

        return config

### Model Core

In [ ]:
#convolutional block
def conv_block(x, filters, dropout=0):
    conv = layers.Conv2D(filters, (3, 3), kernel_initializer='he_normal', padding="same")(x)
    conv = layers.BatchNormalization(axis=3)(conv)
    conv = layers.Activation("relu")(conv)
    if dropout > 0:
        conv = layers.Dropout(dropout)(conv)
    conv = layers.Conv2D(filters, (3, 3), kernel_initializer='he_normal', padding="same")(conv)
    conv = layers.BatchNormalization(axis=3)(conv)
    conv = layers.Activation("relu")(conv)
    return conv

In [ ]:
def ASPPBlock(x, num_filters):
        size = tf.shape(x)[1:3]
        conv_1x1 = layers.Conv2D(num_filters, (1, 1), padding='same', activation='relu')(x)

        conv_3x3_1 = layers.Conv2D(num_filters, (3, 3), padding='same', dilation_rate=6, activation='relu')(x)
        conv_3x3_2 = layers.Conv2D(num_filters, (3, 3), padding='same', dilation_rate=12, activation='relu')(x)
        conv_3x3_3 = layers.Conv2D(num_filters, (3, 3), padding='same', dilation_rate=18, activation='relu')(x)
        conv_3x3_4 = layers.Conv2D(num_filters, (3, 3), padding='same', dilation_rate=24, activation='relu')(x)

        global_avg_pool = layers.GlobalAveragePooling2D()(x)
        global_avg = tf.reshape(global_avg_pool, (-1, 1, 1, tf.shape(global_avg_pool)[-1]))
        global_avg = tf.image.resize(global_avg, size, method='bilinear')

        Concat=layers.Concatenate()([conv_1x1, conv_3x3_1, conv_3x3_2, conv_3x3_3, conv_3x3_4, global_avg])

        conv_global = layers.Conv2D(num_filters, (1, 1), padding='same', activation='relu')(Concat)

        return conv_global

In [ ]:
def SwinAsppUNetModel(window_size=7, embed_dim=96, depths=(2, 2, 6, 2), num_heads=(3, 6, 12, 24),
                   input_shape=(256, 256, 27), patch_size=4, patch_norm=True, use_ape=False, drop_rate=0., mlp_ratio=4.,
                   qkv_bias=True, qk_scale=None, preprocess=None, attn_drop=0., path_drop=0.2, window_pretrain=None,
                   swin_v2=None, model_name='swin_aspp_unet', weights=None, input_tensor=None,
                   pooling=None, dropout=0.1, num_classes=7,pretrain_size=224):



    # Determine proper input shape
    if preprocess:
        input = layers.Input(shape=input_shape, dtype='uint8', name='input_layer')
        image = layers.Lambda(preprocess_input, name='preprocess_layer')(input)
        x = PatchEmbedding(patch_size=patch_size, embed_dim=embed_dim, normalize=patch_norm, name='patch_embed')(image)
    else:
        input = layers.Input(shape=input_shape)
        x = PatchEmbedding(patch_size=patch_size, embed_dim=embed_dim, normalize=patch_norm, name='patch_embed')(input)


    # Swin Transformer Encoder:
    # classes = output_shape[-1]
    x_skip = []

    # Define model pipeline

    x = layers.Dropout(drop_rate, name='pos_drop')(x)

    path_drops = np.linspace(0., path_drop, sum(depths))

    if not swin_v2:
        window_pretrain = np.minimum(window_size, pretrain_size // 2 ** np.arange(2, 6)).tolist()
    elif window_pretrain is None and swin_v2:
        window_pretrain = [0] * len(depths)


    size_for_upsample = embed_dim
    size_of_dim = input_shape[0]//patch_size
    for i in range(len(depths)):
        path_drop = path_drops[sum(depths[:i]):sum(depths[:i + 1])].tolist()
        not_last = i != len(depths) - 1

        x = BasicLayer(depth=depths[i], num_heads=num_heads[i], window_size=window_size, mlp_ratio=mlp_ratio,
                       qkv_bias=qkv_bias, qk_scale=qk_scale, drop=drop_rate, attn_drop=attn_drop, path_drop=path_drop,
                       window_pretrain=window_pretrain[i], swin_v2=swin_v2, name=f'layers.{i}')(x)


        x_skip.append(x)
        if not_last:
            size_for_upsample = size_for_upsample*2
            size_of_dim = size_of_dim//2
            x = PatchMerging(swin_v2=swin_v2, name=f'layers.{i}_downsample')(x)


    x = LayerNorm(name='norm')(x)


    #ASPP encoder:
    ASPPBlock_1=ASPPBlock(input,16)
    pol1= layers.MaxPooling2D(pool_size=(4,4))(ASPPBlock_1) #64*64*16

    ASPPBlock_2=ASPPBlock(pol1,32)
    pol2=layers.MaxPooling2D(pool_size=(2,2))(ASPPBlock_2) #32*32*32

    ASPPBlock_3=ASPPBlock(pol2,64)
    pol3=layers.MaxPooling2D(pool_size=(2,2))(ASPPBlock_3) #16*16*64

    ASPPBlock_4=ASPPBlock(pol3,128)
    pol4=layers.MaxPooling2D(pool_size=(2,2))(ASPPBlock_4) #8*8*128


    E=[pol3,pol2,pol1]
    # Skip connections
    x_skip = x_skip[::-1]
    num_heads = num_heads[::-1]
    depths = depths[::-1]
    depths = depths[:-1]
    X = x_skip[0]

    # Decoder
    filters_d = [256, 128, 64, 32]
    depth_decode = len(E)

    # Start decoding
    X = layers.concatenate([X, pol4], axis=-1)  # Combine ASPPNet and swin encoder outputs in the Bottleneck
    print(X.shape)
    X = conv_block(X, 512)

    x_decode = x_skip[1:]

    filters_d=[256,128,64,32]
    depth_decode = len(x_decode)

    for i in range(depth_decode):

        X = layers.Conv2DTranspose(filters_d[i], (3, 3), strides=(2,2), padding='same')(X)

        X = layers.concatenate([X, x_decode[i], E[i]], axis=-1, name='upsampling_concat_{}'.format(i))

        X = conv_block(X, filters_d[i], dropout)


    X = layers.Conv2DTranspose(filters_d[3], (3, 3), strides=(4,4), padding='same')(X)
    X = conv_block(X, filters_d[3], dropout)
    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(X)

    # Create model.
    model = models.Model(input, outputs, name=model_name)
    print(model.summary())

    return model

In [ ]:
model=SwinAsppUNetModel()

### Work

In [ ]:
width, height,chanel=256,256,27
n_class=7

all_files_loc_train = 'C:/Fuel/NPZ format 256/'
all_files_train = os.listdir(all_files_loc_train)

image_label_map = {
        "Input_{}.npz".format(i+1): "Label_{}.npz".format(i+1)
        for i in range(int(len(all_files_train)/2))}
partition_train = [item for item in all_files_train if "Input" in item]

print('All files:',len(all_files_train))
print('Label files:',len(partition_train))


In [ ]:
class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, list_examples, batch_size=8, dim=(width,height,chanel),shuffle=True):
        # Constructor of the data generator.
        self.dim = dim
        self.batch_size = batch_size
        self.list_examples = list_examples
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        # Denotes the number of batches per epoch
        return int(np.floor(len(self.list_examples) / self.batch_size))

    def __getitem__(self, index):
        # Generate one batch of data
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]

        # Find list of IDs
        list_IDs_temp = [self.list_examples[k] for k in indexes]

        # Generate data
        X, y = self.__data_generation(list_IDs_temp)

        return X, y

    def on_epoch_end(self):
        # This function is called at the end of each epoch.
        self.indexes = np.arange(len(self.list_examples))
        if self.shuffle == True:
            np.random.shuffle(self.indexes)

    def __data_generation(self, list_IDs_temp):
        # Load individual numpy arrays and aggregate them to a batch.

        X = np.empty([self.batch_size, self.dim[0], self.dim[1],self.dim[2]],dtype='float32')

        # y is a one-hot encoded vector.
        y = np.empty([self.batch_size,self.dim[0], self.dim[1],n_class], dtype=np.float32)

        # Generate data.

        c=0
        for i in list_IDs_temp:

            x_file_path = os.path.join(all_files_loc_train, i)
            y_file_path = os.path.join(all_files_loc_train, image_label_map.get(i))

            # Load sample

            X[c, :,:,:] = np.load(x_file_path)['hr']
            # Load labels

            y[c,:,:,:] = np.load(y_file_path)['lr']

            c=c+1

        return X, y


training_generator = DataGenerator(partition_train)

In [ ]:
#Keras
ALPHA = 0.7
BETA = 0.3
def TverskyLoss(targets, inputs, alpha=ALPHA, beta=BETA, smooth=1e-6):

        #flatten label and prediction tensors
    inputs = K.flatten(inputs)
    targets = K.flatten(targets)

        #True Positives, False Positives & False Negatives
    TP = K.sum((inputs * targets))
    FP = K.sum(((1-targets) * inputs))
    FN = K.sum((targets * (1-inputs)))

    Tversky = (TP + smooth) / (TP + alpha*FP + beta*FN + smooth)
    return (1-Tversky)

def iou(y_true, y_pred,smooth=1):
    intersection = K.sum(y_true * y_pred)
    sum_ = K.sum(y_true + y_pred)
    jac = (intersection + smooth) / (sum_ - intersection + smooth)
    return jac

def jac_distance(y_true, y_pred):
    y_truef=K.flatten(y_true)
    y_predf=K.flatten(y_pred)

    return - iou(y_true, y_pred)

In [ ]:
# Confident Area Selection (CAS) module to create masks for CA and VA
def confident_area_selection(predictions, tau):
    """
    Selects the confident and vague areas based on the prediction probabilities.
    
    Args:
        predictions: The predicted probabilities, shape (batch_size, height, width, num_classes)
        tau: Threshold value to split the confident and vague areas.

    Returns:
        ca_mask: Confident area mask (1 for CA, 0 for VA), same shape as predictions.
        va_mask: Vague area mask (0 for CA, 1 for VA), same shape as predictions.
    """
    max_prob = tf.reduce_max(predictions, axis=-1)  # Max probability for each pixel
    ca_mask = tf.cast(max_prob >= tau, tf.float32)  # Confident Area Mask
    va_mask = 1.0 - ca_mask  # Vague Area Mask
    return ca_mask, va_mask

def calculate_threshold(predictions):
    """
    Calculate the dynamic threshold (tau) as the average of maximum class probabilities
    across all pixels in the batch.
    
    Args:
        predictions: Predicted probabilities, shape (batch_size, height, width, num_classes)
        
    Returns:
        tau: Dynamic threshold value.
    """
    max_probs = tf.reduce_max(predictions, axis=-1)  # Max probability for each pixel
    tau = tf.reduce_mean(max_probs)  # Mean of the max probabilities
    return tau

def get_predicted_classes_numpy(y_pred):
    """
    Returns the predicted class for each pixel as a NumPy array.
    
    Args:
        y_pred: Predicted probabilities from the model, shape (batch_size, height, width, num_classes)
        
    Returns:
        predicted_classes_numpy: The predicted class for each pixel as a NumPy array, shape (batch_size, height, width)
    """
    predicted_classes = tf.argmax(y_pred, axis=-1)
    predicted_classes_numpy = predicted_classes.numpy()  # Convert to NumPy array
    return predicted_classes_numpy


def l2h_loss(y_true, y_pred, class_weights, gamma=0.05, iou_weight=0.1):
    """
    L2H Loss function with class weights, an additional IoU loss component, and DVA loss for softmax outputs.
    
    Args:
        y_true: True labels (batch_size, height, width, num_classes)
        y_pred: Predicted probabilities (batch_size, height, width, num_classes)
        class_weights: List or tensor of class weights, with one weight per class.
        gamma: Scaling factor for the DVA loss.
        iou_weight: Scaling factor for the IoU loss.
    
    Returns:
        Total L2H loss including weighted cross-entropy, DVA loss, and IoU loss.
    """
    
    # Convert class weights to a tensor
    class_weights_tensor = tf.constant(class_weights, dtype=tf.float32)

    # Calculate dynamic threshold
    tau = calculate_threshold(y_pred)
    tau = 0.5  # or dynamically set `tau`
    
    # Calculate confident and vague areas
    ca_mask, va_mask = confident_area_selection(y_pred, tau)
    
    # Expand ca_mask and va_mask to match y_pred dimensions
    ca_mask_expanded = tf.expand_dims(ca_mask, axis=-1)
    va_mask_expanded = tf.expand_dims(va_mask, axis=-1)
    
    # Weighted Cross-entropy loss over confident area
    ce_loss = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
    weighted_ce_loss = tf.reduce_mean(ce_loss * tf.reduce_sum(y_true * class_weights_tensor, axis=-1))
    ce_loss_confident = tf.reduce_mean(weighted_ce_loss * tf.squeeze(ca_mask_expanded, axis=-1))
    
    # Dynamic Vague Area (DVA) loss
    pooled_ca_features = tf.reduce_mean(y_pred * ca_mask_expanded, axis=[1, 2])
    pooled_va_features = tf.reduce_mean(y_pred * va_mask_expanded, axis=[1, 2])
    variance = tf.reduce_mean(tf.square(pooled_ca_features - pooled_va_features))
    dva_loss = gamma * variance
    
    # Convert softmax predictions to one-hot encoded labels
    y_pred_one_hot = tf.one_hot(tf.argmax(y_pred, axis=-1), depth=y_pred.shape[-1])
    
    # IoU Loss
    intersection = tf.reduce_sum(y_true * y_pred_one_hot, axis=[1, 2])
    union = tf.reduce_sum(y_true + y_pred_one_hot, axis=[1, 2]) - intersection
    iou_loss = tf.reduce_mean(1 - (intersection + 1e-7) / (union + 1e-7))  # Adding epsilon to avoid division by zero
    
    # Total L2H loss
    total_loss = ce_loss_confident + dva_loss + iou_weight * iou_loss
    
    return 10 * total_loss


In [ ]:
# Confident Area Selection (CAS) module to create masks for CA and VA
def confident_area_selection(predictions, tau):
    """
    Selects the confident and vague areas based on the prediction probabilities.
    
    Args:
        predictions: The predicted probabilities, shape (batch_size, height, width, num_classes)
        tau: Threshold value to split the confident and vague areas.

    Returns:
        ca_mask: Confident area mask (1 for CA, 0 for VA), same shape as predictions.
        va_mask: Vague area mask (0 for CA, 1 for VA), same shape as predictions.
    """
    max_prob = tf.reduce_max(predictions, axis=-1)  # Max probability for each pixel
    ca_mask = tf.cast(max_prob >= tau, tf.float32)  # Confident Area Mask
    va_mask = 1.0 - ca_mask  # Vague Area Mask
    return ca_mask, va_mask

def calculate_threshold(predictions):
    """
    Calculate the dynamic threshold (tau) as the average of maximum class probabilities
    across all pixels in the batch.
    
    Args:
        predictions: Predicted probabilities, shape (batch_size, height, width, num_classes)
        
    Returns:
        tau: Dynamic threshold value.
    """
    max_probs = tf.reduce_max(predictions, axis=-1)  # Max probability for each pixel
    tau = tf.reduce_mean(max_probs)  # Mean of the max probabilities
    return tau

def get_predicted_classes_numpy(y_pred):
    """
    Returns the predicted class for each pixel as a NumPy array.
    
    Args:
        y_pred: Predicted probabilities from the model, shape (batch_size, height, width, num_classes)
        
    Returns:
        predicted_classes_numpy: The predicted class for each pixel as a NumPy array, shape (batch_size, height, width)
    """
    predicted_classes = tf.argmax(y_pred, axis=-1)
    predicted_classes_numpy = predicted_classes.numpy()  # Convert to NumPy array
    return predicted_classes_numpy


def l2h_loss(y_true, y_pred, class_weights=[1.08025321,1.11074909,1.06996478,1.08056692,1.16622461,1.3762863,1.11595514],
             gamma=0.05, iou_weight=0.1):
    """
    L2H Loss function with class weights, an additional IoU loss component, and DVA loss for softmax outputs.
    
    Args:
        y_true: True labels (batch_size, height, width, num_classes)
        y_pred: Predicted probabilities (batch_size, height, width, num_classes)
        class_weights: List or tensor of class weights, with one weight per class.
        gamma: Scaling factor for the DVA loss.
        iou_weight: Scaling factor for the IoU loss.
    
    Returns:
        Total L2H loss including weighted cross-entropy, DVA loss, and IoU loss.
    """
    
    # Convert class weights to a tensor
    class_weights_tensor = tf.constant(class_weights, dtype=tf.float32)

    # Calculate dynamic threshold
    tau = calculate_threshold(y_pred)
    # tau = 0.5  # or dynamically set `tau`
    
    # Calculate confident and vague areas
    ca_mask, va_mask = confident_area_selection(y_pred, tau)
    
    # Expand ca_mask and va_mask to match y_pred dimensions
    ca_mask_expanded = tf.expand_dims(ca_mask, axis=-1)
    va_mask_expanded = tf.expand_dims(va_mask, axis=-1)
    
    # Weighted Cross-entropy loss over confident area
    ce_loss = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
    weighted_ce_loss = tf.reduce_mean(ce_loss * tf.reduce_sum(y_true * class_weights_tensor, axis=-1))
    ce_loss_confident = tf.reduce_mean(weighted_ce_loss * tf.squeeze(ca_mask_expanded, axis=-1))
    
    # Dynamic Vague Area (DVA) loss
    pooled_ca_features = tf.reduce_mean(y_pred * ca_mask_expanded, axis=[1, 2])
    pooled_va_features = tf.reduce_mean(y_pred * va_mask_expanded, axis=[1, 2])
    variance = tf.reduce_mean(tf.square(pooled_ca_features - pooled_va_features))
    dva_loss = gamma * variance
    
    # Convert softmax predictions to one-hot encoded labels
    y_pred_one_hot = tf.one_hot(tf.argmax(y_pred, axis=-1), depth=y_pred.shape[-1])
    
    # IoU Loss
    intersection = tf.reduce_sum(y_true * y_pred_one_hot, axis=[1, 2])
    union = tf.reduce_sum(y_true + y_pred_one_hot, axis=[1, 2]) - intersection
    iou_loss = tf.reduce_mean(1 - (intersection + 1e-7) / (union + 1e-7))  # Adding epsilon to avoid division by zero
    
    # Total L2H loss
    total_loss = ce_loss_confident + dva_loss + iou_weight * iou_loss
    
    return 10 * total_loss


In [ ]:
class_weights_list=[1.1885373319585373,
                     1.1387892969753706,
                     1.2167463730689725,
                     1.1909241956842624,
                     1.09399307888407,
                     1.0395509767958888,
                     1.1314587466328985]

# Define the custom L2H loss function with class weights
def l2h_loss_with_weights(y_true, y_pred):
    return l2h_loss(y_true, y_pred, class_weights=class_weights_list)


# model = Model_((width, height, chanel), n_class)
# Set the initial learning rate
initial_learning_rate = 0.0005

# Optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=initial_learning_rate)
# Compile the model
model.compile(optimizer=optimizer, loss=l2h_loss_with_weights, metrics=[iou,'accuracy'])

# Implement ReduceLROnPlateau
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='accuracy',        # Monitor loss to reduce learning rate when it stops improving
    factor=0.5,               # Reduce the learning rate by a factor of 10
    patience=5,               # Number of epochs with no improvement after which learning rate will be reduced
    min_lr=1e-10,              # Lower bound on the learning rate
    verbose=1                 # Print the message when learning rate is reduced
)
checkpoint_filepath='C:/Fuel/Swin256/'
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    checkpoint_filepath,
    monitor="accuracy",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

# Define EarlyStopping callback
early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='accuracy',          # Metric to monitor (e.g., 'accuracy' or 'val_loss')
    patience=25,                 # Number of epochs with no improvement to stop training
    restore_best_weights=True,   # Restore model weights from the epoch with the best monitored metric
    verbose=1                    # Print message when stopping
)

callbacksh5 = keras.callbacks.ModelCheckpoint('C:/Fuel/Models/model.h5',monitor='loss',
                             verbose=1, save_best_only=True)
# Train the model with the learning rate scheduler
history = model.fit(
    training_generator,
    epochs=1000,
    callbacks=[lr_scheduler, checkpoint_callback,early_stopping_callback]
)



In [ ]:
np.save('loss.npy',history.history['loss'])
np.save('iou.npy',history.history['iou'])
np.save('accuracy.npy',history.history['accuracy'])
